# Block 2: Local Structures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatge-upc/aa2/blob/main/notebooks/aa2_2_1_cnn_fundamental_layers.ipynb)

## 2.1: CNNs: Fundamental Layers & Architectures

Short PyTorch demonstrations to accompany the slides:

* **Convolution and pooling**: Inspect local operations, channels, padding and stride.
* **A small CNN**: Follow the tensor shapes from an image batch to class logits.
* **MNIST, transforms and VGG16**: Load and resize handwritten digits, run a forward pass through the small CNN, and inspect the VGG16 architecture and its intermediate shapes.

The examples run on CPU. MNIST is downloaded into `./data` on the first run and reused afterwards.

**Before running:** predict the shapes or values asked below. This simple exercise helps you understand how the fundamental CNN parameters—channels, kernel size, padding and stride—determine the output shapes and values. Replace the `(0, 0, 0, 0)` placeholders with your predicted shape tuples before executing. `check_shape` reveals the output shape and tensor only when your prediction matches. Otherwise, revise your prediction and try again. 

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(7)
torch.set_printoptions(precision=2, sci_mode=False)


def check_shape(prediction, result):
    actual_shape = tuple(result.shape)
    print("\nOutput:")
    if prediction is None:
        print("Shape:", actual_shape)
        print("Output tensor:\n", result)
        return True
    elif prediction == actual_shape:
        print("✅ Matches your prediction:", actual_shape)
        print("Output tensor:\n", result)
        return True
    else:
        print(f"🔄 Different from your prediction: you predicted {prediction}. Try again!")
        return False

### 1. Convolution: A Filter (Kernel) Slides Across the Input

*Images* have shape **`(N, C, H, W)`**: batch size, channels, height and width. Here we use one *image* with one channel and a 4×4 grid of values.

[`nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) slides a filter (kernel) across the input image. At each position, it multiplies the kernel coefficients by the corresponding input values and sums the products to produce one output value. The same coefficients are reused at every position. We set its 2×2 weights to `1/4` to compute a **local average**, with no bias. These are hand-set demonstration weights; CNN training learns the weights.

The input contains the integers 1–16 in row order. The stride is 1 and there is no padding.

**Predict before running:** What is the output shape, including batch and channels? What is the top-left output value?

In [ ]:
output_shape = (0, 0, 0, 0)  # Fill the shape tuple for output feature_map, including batch and channels.

x = torch.arange(1, 17, dtype=torch.float32).reshape(1, 1, 4, 4)
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=2, bias=False)

with torch.no_grad():
    conv.weight.fill_(0.25)
    feature_map = conv(x)

print("Input shape:", tuple(x.shape))
print(x[0, 0])
print("\nKernel:")
print(conv.weight[0, 0])
ok = check_shape(output_shape, feature_map)

#### Multiple Input Channels: Each Filter Uses Them All

For a standard convolution (`groups=1`, the default), **each output filter has a separate spatial kernel for every input channel**. Their contributions are summed into one output feature map. Different output filters have different weights.

Here channel 0 contains ones and channel 1 contains tens. We make two output filters, each with two 2×2 kernels. The weight tensor has shape **`(out_channels, in_channels, kH, kW)`**.

* Filter 0: kernel values 1 for channel 0 and 2 for channel 1.
* Filter 1: kernel values 3 for channel 0 and 4 for channel 1.

Each filter spans **both input channels**, and produces **one** output channel.

**Predict before running:** How many output feature maps do we get, and what is their full tensor shape? What is the first output value of each filter? Work out each input channel's contribution before summing.

In [ ]:
output_shape = (0, 0, 0, 0)  # Fill a shape tuple, including batch and channels.

multi_input = torch.cat([
    torch.ones(1, 1, 3, 3),
    10 * torch.ones(1, 1, 3, 3),
], dim=1)  # Shape: (1, 2, 3, 3)
multi_conv = nn.Conv2d(2, 2, kernel_size=2, bias=False)

with torch.no_grad():
    multi_conv.weight[0, 0].fill_(1)  # Output filter 0, input channel 0
    multi_conv.weight[0, 1].fill_(2)  # Output filter 0, input channel 1
    multi_conv.weight[1, 0].fill_(3)  # Output filter 1, input channel 0
    multi_conv.weight[1, 1].fill_(4)  # Output filter 1, input channel 1
    multi_output = multi_conv(multi_input)

print("Input shape:", tuple(multi_input.shape))
print("Weight shape:", tuple(multi_conv.weight.shape))
print("Filter 0 kernels:\n", multi_conv.weight[0])
print("Filter 1 kernels:\n", multi_conv.weight[1])
ok = check_shape(output_shape, multi_output)
if ok:
    print(f"Let's verify first coeficient: {multi_output[0,0,0,0].item()=}")
    print(f"First sliding window:\n{multi_input[0, :, :2, :2]}")
    contributions = (multi_input[0, :, :2, :2] *
                     multi_conv.weight[0].detach()).sum(dim=(1, 2))
    print("Filter 0, first window: channel contributions =", contributions)
    print("Sum =", contributions.sum().item())

#### Padding and Stride

We reuse the 4×4 input with the **same 3×3 averaging kernel** in all three cases:

* `padding=0, stride=1`: only windows fully inside the image.
* `padding=1, stride=1`: add a border of zeros.
* `padding=1, stride=2`: move the window two pixels at a time.

Padding lets us control the spatial size; increasing stride downsamples the feature map. The averages at padded boundaries include zeros.

**Predict before running:** What is the output shape for each setting? Which setting preserves the input height and width?

In [ ]:
output_shapes = {  # Fill a shape tuple for each (padding, stride) setting.
    (0, 1): (0, 0, 0, 0),
    (1, 1): (0, 0, 0, 0),
    (1, 2): (0, 0, 0, 0),
}

for padding, stride in [(0, 1), (1, 1), (1, 2)]:
    conv_setting = nn.Conv2d(1, 1, kernel_size=3, padding=padding,
                             stride=stride, bias=False)
    with torch.no_grad():
        conv_setting.weight.fill_(1 / 9)
        setting_output = conv_setting(x)

    print(f"padding={padding}, stride={stride}")
    check_shape(output_shapes[(padding, stride)], setting_output)
    print()

### 2. Pooling: Maximum or Average

[`nn.MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) keeps the largest value in each window; [`nn.AvgPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html) keeps its mean. Pooling has no learnable weights.

We use 2×2 windows with stride 2 on the same 4×4 image. Both operations preserve the batch and channel dimensions.

**Predict before running:** What is the output shape for each operation? What are the top-left max and average values?

In [ ]:
max_output_shape = (0, 0, 0, 0)  # Fill a shape tuple for max pooling.
avg_output_shape = (0, 0, 0, 0)  # Fill a shape tuple for average pooling.

max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

max_values = max_pool(x)
avg_values = avg_pool(x)

print("Max pooling:")
check_shape(max_output_shape, max_values)
print("\nAverage pooling:")
check_shape(avg_output_shape, avg_values)

### 3. A Complete Small CNN

This **LeNet-style** network uses ReLU and max pooling. It illustrates the architecture rather than reproducing historical LeNet-5 exactly.

In `Conv2d(1, 6, 5)`, `1` is the number of input channels, `6` is the number of output feature maps and `5` is the kernel size. The second convolution combines the six input channels to produce sixteen feature maps.

[`nn.Flatten`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Flatten.html) preserves the batch dimension and turns each sample's feature maps into a vector. The linear layers produce one logit per digit class.

**Predict before running:** For an input of shape `(8, 1, 32, 32)`, what is the final output shape? How many values per sample enter the first linear layer?

In [ ]:
output_shape = (0, 0)  # Fill the final logits shape tuple.

model = nn.Sequential(                  # Input: N × 1 × 32 × 32
    nn.Conv2d(1, 6, kernel_size=5),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(6, 16, kernel_size=5),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),
    nn.Linear(400, 120), nn.ReLU(),
    nn.Linear(120, 84), nn.ReLU(),
    nn.Linear(84, 10),
)

x_batch = torch.randn(8, 1, 32, 32)
with torch.inference_mode():
    logits = model(x_batch)

print("Model", model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")
for name, p in model.named_parameters():
    print(f"Layer {name}: {p.numel():,}")

print("\nInput shape:", tuple(x_batch.shape))
ok = check_shape(output_shape, logits)


**Predict the intermediate shapes:** Write down the output shape after every layer, including ReLU and pooling.

The next cell defaults to `show_shapes = False`: it lists each layer's index and description. Write down the shape at the output of the layer. Set it to `True` and rerun to reveal the shapes and compare with your predictions.

In [ ]:
show_shapes = False  # Set True to reveal the shapes after making your predictions.

with torch.inference_mode():
    h = x_batch
    for index, layer in enumerate(model):
        h = layer(h)
        description = f"{index:2d}: {layer}"
        if show_shapes:
            print(f"{description} → {tuple(h.shape)}")
        else:
            print(f"{description} → shape?")


The `torchvision` package provides easy access to image datasets, model architectures, pretrained weights, and utilities. These resources make it easy to gain **hands-on experience** with the concepts covered in the course, including architectures, optimizers, transfer learning, and regularization.


### 4. MNIST: Dataset → Transforms → Mini-Batch

A torchvision [`dataset`](https://docs.pytorch.org/vision/stable/datasets.html) returns an image and its label. [`transforms`](https://docs.pytorch.org/vision/stable/transforms.html) preprocess each image; a `DataLoader` groups samples into mini-batches.

We use [`MNIST`](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html), whose labels are digits 0–9. `Resize` changes its 28×28 grayscale images to 32×32 to match our CNN. `ToTensor` converts each PIL image to a float tensor of shape `(1, 32, 32)`, with pixel values in `[0, 1]`.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

dataset = datasets.MNIST(root="./data", train=False, download=True,
                         transform=transform)
loader = DataLoader(dataset, batch_size=8, shuffle=False)
images, labels = next(iter(loader))

print("Dataset size:", len(dataset))
print("Images:", tuple(images.shape))
print("Labels:", labels.tolist())

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, image, label in zip(axes, images, labels):
    ax.imshow(image[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Label: {label.item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

The same CNN accepts this real image batch. `model.eval()` selects evaluation mode; `torch.inference_mode()` disables gradient tracking for the forward pass.

The weights are randomly initialized, so the scores do not yet represent a learned digit classifier. The output contains raw logits, as expected by `CrossEntropyLoss`.

In [ ]:
model.eval()
with torch.inference_mode():
    logits = model(images)

print("MNIST input:", tuple(images.shape))
print("Class logits:", tuple(logits.shape))
print("First image logits:", logits[0])

### 5. Torchvision Models: Inspect VGG16

[`torchvision.models.vgg16`](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vgg16.html) provides a ready-made architecture. Its `features` stack 3×3 convolutions, ReLU activations and max pooling; its `classifier` contains fully connected layers.

We use `weights=None` to initialize the model randomly and inspect its architecture without downloading pretrained weights. We then pass a synthetic RGB tensor through the network to follow its shapes.

In [ ]:
from torchvision.models import vgg16

vgg = vgg16(weights=None)
print(vgg)
num_params_total = sum(p.numel() for p in vgg.parameters())
num_params_features = sum(p.numel() for p in vgg.features.parameters())
num_params_avgpool = sum(p.numel() for p in vgg.avgpool.parameters()) 
num_params_classifier = sum(p.numel() for p in vgg.classifier.parameters()) 
print(f"Total Trainable parameters: {num_params_total:,}")
print(f"\tFeatures:   {num_params_features/num_params_total*100:5.2f}% ({num_params_features:,})")
print(f"\tAvgPool:    {num_params_avgpool/num_params_total*100:5.2f}% ({num_params_avgpool:,})")
print(f"\tClassifier: {num_params_classifier/num_params_total*100:5.2f}% ({num_params_classifier:,})")

**Predict before running:** For an input of shape `(1, 3, 224, 224)`, how do the spatial dimensions and channels change through the convolution and pooling blocks? What is the shape after flattening and after the last linear layer?

The next cell prints the output shape after every layer. We use evaluation mode and disable gradient tracking; the weights remain randomly initialized.

In [ ]:
vgg_input = torch.randn(1, 3, 224, 224)
vgg.eval()

# Follow VGG16's forward pass to craete a list with all the layers:
#    → features → avgpool → flatten → classifier.
layers = (
    [(f"features.{i}", layer) for i, layer in enumerate(vgg.features)]
    + [("avgpool", vgg.avgpool), ("flatten", nn.Flatten(start_dim=1))]
    + [(f"classifier.{i}", layer) for i, layer in enumerate(vgg.classifier)]
)

print("Input shape:", tuple(vgg_input.shape))
with torch.inference_mode():
    h = vgg_input
    for name, layer in layers:
        h = layer(h)
        description = f"{name}: {layer}"
        print(f"{description} → {tuple(h.shape)}")

### PyTorch Layer Reference

The same layer families work with one, two or three spatial dimensions:

| Operation | 1D: `(N, C, L)` | 2D: `(N, C, H, W)` | 3D: `(N, C, D, H, W)` |
|---|---|---|---|
| Convolution | [`nn.Conv1d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv1d.html) | [`nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) | [`nn.Conv3d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv3d.html) |
| Max pooling | [`nn.MaxPool1d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool1d.html) | [`nn.MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) | [`nn.MaxPool3d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool3d.html) |
| Average pooling | [`nn.AvgPool1d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AvgPool1d.html) | [`nn.AvgPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html) | [`nn.AvgPool3d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AvgPool3d.html) |

[`torch.flatten(x, start_dim=1)`](https://docs.pytorch.org/docs/stable/generated/torch.flatten.html) is the function equivalent of `nn.Flatten()` used above. Other pooling options include [`nn.AdaptiveAvgPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AdaptiveAvgPool2d.html), which lets us specify the output spatial size.